---
## PART 10 — SCLERP: SCREW LINEAR INTERPOLATION

**ScLERP** generates the geometrically optimal path between two poses.
It interpolates along a **screw motion** — the natural rigid body motion
that simultaneously rotates and translates along a single axis.

### Formula
$$\hat{Q}(t) = \hat{Q}_A \otimes (\hat{Q}_A^{-1} \otimes \hat{Q}_B)^t \qquad t \in [0,1]$$

### DQ Power via Screw Parameters
$$\hat{Q}^t = \exp(t \cdot \log(\hat{Q}))$$

### Minimum Jerk Time Parameterisation
$$s(\tau) = 10\tau^3 - 15\tau^4 + 6\tau^5, \qquad \tau = t/T$$

Properties: $s(0)=0$, $s(1)=1$, $\dot{s}(0)=\dot{s}(1)=0$, $\ddot{s}(0)=\ddot{s}(1)=0$

This gives **zero velocity AND acceleration** at start and end — critical
for smooth motor commands without jerks.


In [2]:
# ============================================================
# SCLERP — SCREW LINEAR INTERPOLATION
# ============================================================

def dq_power(Q, t):
    """
    Compute Q^t for a unit dual quaternion via screw decomposition.
    
    For unit DQ Q = q_r + eps*q_d:
      Extract screw parameters (angle theta, pitch d, axis omega, moment m)
      Scale by t
      Reconstruct DQ
    
    Parameters
    ----------
    Q : (8,) unit dual quaternion
    t : float  power parameter in [0,1]
    
    Returns
    -------
    (8,) dual quaternion Q^t
    """
    qr = Q[:4]
    qd = Q[4:]
    
    # Handle double cover
    if qr[0] < 0:
        qr, qd = -qr, -qd
    
    qrw = float(np.clip(qr[0], -1.0, 1.0))
    half_angle = np.arccos(qrw)   # theta/2
    
    if half_angle < 1e-10:
        # Near identity: linear interpolation of translation only
        qd_t = qd * t
        Q_t  = np.concatenate([[1., 0., 0., 0.], qd_t])
        return dq_normalize(Q_t)
    
    sin_ha    = np.sin(half_angle)
    omega_hat = qr[1:] / sin_ha    # unit rotation axis
    
    # Screw pitch: d = -2 * qdw / sin(half_angle)
    d_half  = -qd[0] / sin_ha
    
    # Moment: perpendicular component
    moment = (qd[1:] - d_half * qrw * omega_hat) / sin_ha
    
    # Scale screw parameters by t
    ha_t    = half_angle * t
    d_t_ha  = d_half * t
    
    c_t = np.cos(ha_t)
    s_t = np.sin(ha_t)
    
    # Reconstruct scaled DQ
    qr_t = np.array([c_t,
                      s_t * omega_hat[0],
                      s_t * omega_hat[1],
                      s_t * omega_hat[2]])
    
    qd_t = np.array([
        -d_t_ha * s_t,
         d_t_ha * c_t * omega_hat[0] + s_t * moment[0],
         d_t_ha * c_t * omega_hat[1] + s_t * moment[1],
         d_t_ha * c_t * omega_hat[2] + s_t * moment[2]
    ])
    
    return dq_normalize(np.concatenate([qr_t, qd_t]))


def sclerp(Q_A, Q_B, t):
    """
    Screw Linear Interpolation between poses Q_A and Q_B.
    
    Q(t) = Q_A ⊗ (Q_A^{-1} ⊗ Q_B)^t
    
    At t=0: returns Q_A exactly
    At t=1: returns Q_B exactly
    Traces the shortest screw motion between the two poses.
    
    Parameters
    ----------
    Q_A, Q_B : (8,) dual quaternions
    t        : float  parameter in [0,1]
    
    Returns
    -------
    (8,) interpolated dual quaternion
    """
    # Relative motion from A to B
    Q_rel = dq_multiply(dq_conjugate(Q_A), Q_B)
    
    # Handle double cover for relative motion
    if Q_rel[0] < 0:
        Q_rel = -Q_rel
    
    # Interpolate relative motion
    Q_rel_t = dq_power(Q_rel, t)
    
    # Apply to start pose
    return dq_normalize(dq_multiply(Q_A, Q_rel_t))


def min_jerk(tau):
    """
    Minimum jerk trajectory profile: s(tau) = 10t^3 - 15t^4 + 6t^5
    tau in [0,1], returns s in [0,1]
    Zero velocity and acceleration at start and end.
    """
    return 10*tau**3 - 15*tau**4 + 6*tau**5


def generate_trajectory(Q_start, Q_end, n_points=250):
    """
    Generate trajectory from Q_start to Q_end using ScLERP
    with minimum-jerk time parameterisation.
    
    Parameters
    ----------
    Q_start  : (8,)  start pose DQ
    Q_end    : (8,)  end pose DQ
    n_points : int   number of trajectory points (250 = 2s at 125Hz)
    
    Returns
    -------
    trajectory : list of (8,) DQ poses
    s_values   : (n_points,) time parameter values
    """
    s_vals = np.array([min_jerk(i / (n_points-1)) for i in range(n_points)])
    trajectory = [sclerp(Q_start, Q_end, s) for s in s_vals]
    return trajectory, s_vals


# ============================================================
# Verify ScLERP
# ============================================================
print('=== ScLERP Verification ===')
print()

theta_A = np.array([0.0, -np.pi/2, np.pi/2, -np.pi/2, -np.pi/2, 0.0])
theta_B = np.array([np.pi/4, -np.pi/3, np.pi/3, -np.pi/4, -np.pi/3, np.pi/6])

Q_A = fk_dq(S, theta_A, M_dq)
Q_B = fk_dq(S, theta_B, M_dq)

# Verify endpoints
Q_t0 = sclerp(Q_A, Q_B, 0.0)
Q_t1 = sclerp(Q_A, Q_B, 1.0)

err_start = np.max(np.abs(Q_t0 - dq_normalize(Q_A)))
err_end   = np.max(np.abs(Q_t1 - dq_normalize(Q_B)))
print(f'ScLERP(t=0) == Q_A: max error = {err_start:.2e}  ', '✓' if err_start < 1e-10 else '✗')
print(f'ScLERP(t=1) == Q_B: max error = {err_end:.2e}   ', '✓' if err_end < 1e-10 else '✗')

# Generate full trajectory
traj, s_vals = generate_trajectory(Q_A, Q_B, n_points=250)

# Extract positions for plotting
positions = np.array([dq_translation(q) for q in traj])

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot(positions[:,0], positions[:,1], positions[:,2], 'b-', linewidth=2)
ax1.scatter(*positions[0], color='green', s=100, zorder=5, label='Start')
ax1.scatter(*positions[-1], color='red', s=100, zorder=5, label='End')
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)'); ax1.set_zlabel('Z (m)')
ax1.set_title('End-Effector Screw Path (ScLERP)')
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2)
tau_vals = np.linspace(0, 1, 250)
ax2.plot(tau_vals, s_vals, 'b-', linewidth=2, label='s(τ) position')
ds = np.gradient(s_vals, tau_vals)
ax2.plot(tau_vals, ds/ds.max(), 'r--', linewidth=2, label='ṡ(τ) velocity (norm)')
dds = np.gradient(ds, tau_vals)
ax2.plot(tau_vals, dds/np.max(np.abs(dds)), 'g:', linewidth=2, label='s̈(τ) accel (norm)')
ax2.set_xlabel('τ = t/T'); ax2.set_ylabel('Normalised value')
ax2.set_title('Min-Jerk Time Parameterisation')
ax2.legend(); ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print(f'\nTrajectory: {len(traj)} poses generated')
print(f'Path length: {np.sum(np.linalg.norm(np.diff(positions, axis=0), axis=1))*1000:.2f} mm')

=== ScLERP Verification ===



NameError: name 'np' is not defined

---
## PART 11 — COMPLETE MOTION PLANNING PIPELINE

Now we chain everything together:

```
1. Define start/end poses in DQ space
2. ScLERP + min-jerk → 250 Cartesian DQ poses
3. IK at each pose → 250 joint angle sets
4. (Simulate) send to motors at 125 Hz
```

**Key insight:** Each IK call is warm-started from the **previous solution**,
so convergence typically takes only 1-5 iterations (not 100+).
This makes online trajectory execution feasible at 125 Hz.


In [ ]:
# ============================================================
# COMPLETE MOTION PLANNING PIPELINE
# ============================================================

def plan_trajectory(S_axes, M_dq, theta_start, Q_end,
                    n_points=100, ik_tol=1e-8, verbose=True):
    """
    Plan a trajectory from current joint config to target DQ pose.
    
    Steps:
    1. FK from theta_start -> Q_start
    2. ScLERP(Q_start, Q_end, t) -> waypoints
    3. IK at each waypoint (warm-started from previous)
    
    Parameters
    ----------
    S_axes      : (6,6)  screw axes
    M_dq        : (8,)   home DQ
    theta_start : (6,)   starting joint angles
    Q_end       : (8,)   target end-effector DQ
    n_points    : int    number of trajectory waypoints
    ik_tol      : float  IK convergence tolerance per waypoint
    
    Returns
    -------
    theta_traj  : (n_points, 6)  joint angle trajectory
    pose_traj   : list of (8,) DQ poses along path
    ik_errors   : (n_points,) IK error at each point
    """
    # Step 1: Start pose from current joints
    Q_start = fk_dq(S_axes, theta_start, M_dq)
    
    # Step 2: Generate ScLERP trajectory
    pose_traj, s_vals = generate_trajectory(Q_start, Q_end, n_points=n_points)
    
    # Step 3: IK at each waypoint
    theta_traj = np.zeros((n_points, 6))
    ik_errors  = np.zeros(n_points)
    theta_traj[0] = theta_start
    
    theta_prev = theta_start.copy()
    
    for i, Q_wp in enumerate(pose_traj):
        # Warm start from previous solution
        theta_sol, conv, hist = inverse_kinematics(
            S_axes, M_dq, Q_wp, theta_prev,
            tol=ik_tol, max_iter=500,
            lam=0.05, alpha=0.5   # tighter params for warm-started IK
        )
        theta_traj[i] = theta_sol
        
        # Measure actual FK error at this point
        Q_actual = fk_dq(S_axes, theta_sol, M_dq)
        pos_e, rot_e = pose_error_metrics(Q_actual, Q_wp)
        ik_errors[i] = pos_e * 1000   # mm
        
        theta_prev = theta_sol.copy()  # warm start next iter
        
        if verbose and i % 20 == 0:
            print(f'  Waypoint {i:3d}/{n_points}: pos_err={pos_e*1000:.3f}mm  '
                  f'rot_err={np.degrees(rot_e):.3f}deg  conv={conv}')
    
    return theta_traj, pose_traj, ik_errors


# ============================================================
# Run complete trajectory
# ============================================================
print('Planning trajectory...')
print()

s_pos = np.random.uniform(-np.pi*0.2, np.pi*0.2, 6)
#theta_start = np.array([0.0, -np.pi/2, np.pi/2, -np.pi/2, -np.pi/2, 0.0])
theta_start = s_pos
e_pos = np.random.uniform(-np.pi*0.2, np.pi*0.2, 6)
#theta_end   = np.array([np.pi/3, -np.pi/4, np.pi/4, -np.pi/3, -np.pi/4, np.pi/6])
theta_end = e_pos
Q_end_target = fk_dq(S, theta_end, M_dq)

theta_traj, pose_traj, ik_errors = plan_trajectory(
    S, M_dq, theta_start, Q_end_target,
    n_points=200, ik_tol=1e-8, verbose=True
)

# ============================================================
# Plot trajectory results
# ============================================================
positions_traj = np.array([dq_translation(q) for q in pose_traj])
time_axis = np.linspace(0, 2.0, len(pose_traj))   # 2 second motion

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Joint angles over time
labels_j = ['Joint 1','Joint 2','Joint 3','Joint 4','Joint 5','Joint 6']
for j in range(6):
    axes[0,0].plot(time_axis, np.degrees(theta_traj[:, j]), linewidth=2, label=labels_j[j])
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_ylabel('Joint Angle (deg)')
axes[0,0].set_title('Joint Angle Trajectory')
axes[0,0].legend(loc='upper right', fontsize=8); axes[0,0].grid(True, alpha=0.4)

# End-effector position over time
axes[0,1].plot(time_axis, positions_traj[:,0]*1000, 'r-', linewidth=2, label='X')
axes[0,1].plot(time_axis, positions_traj[:,1]*1000, 'g-', linewidth=2, label='Y')
axes[0,1].plot(time_axis, positions_traj[:,2]*1000, 'b-', linewidth=2, label='Z')
axes[0,1].set_xlabel('Time (s)'); axes[0,1].set_ylabel('Position (mm)')
axes[0,1].set_title('End-Effector Position Trajectory')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.4)

# 3D path
ax3d = fig.add_subplot(2, 2, 3, projection='3d')
ax3d.plot(positions_traj[:,0], positions_traj[:,1], positions_traj[:,2],
           'b-', linewidth=2)
ax3d.scatter(*positions_traj[0],  color='green', s=100, label='Start')
ax3d.scatter(*positions_traj[-1], color='red',   s=100, label='End')
ax3d.set_xlabel('X(m)'); ax3d.set_ylabel('Y(m)'); ax3d.set_zlabel('Z(m)')
ax3d.set_title('3D End-Effector Path')
ax3d.legend()

# IK errors along trajectory
axes[1,1].plot(time_axis, ik_errors, 'r-', linewidth=2)
axes[1,1].set_xlabel('Time (s)'); axes[1,1].set_ylabel('Position Error (mm)')
axes[1,1].set_title('IK Tracking Error Along Trajectory')
axes[1,1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print(f'\nTrajectory Statistics:')
print(f'  Waypoints:            {len(pose_traj)}')
print(f'  Mean IK error:        {ik_errors.mean():.4f} mm')
print(f'  Max IK error:         {ik_errors.max():.4f} mm')
print(f'  Path length:          {np.sum(np.linalg.norm(np.diff(positions_traj, axis=0), axis=1))*1000:.2f} mm')

---
## PART 13 — MOTOR COMMAND PIPELINE (SIMULATION)

This simulates what would run on real hardware at **125 Hz** (every 8ms).

The control loop:
```
READ encoders -> theta_actual
LOOKUP theta_ref(t) from precomputed trajectory
COMPUTE error: e = theta_ref - theta_actual
PID -> torque command tau
CONVERT tau -> motor current I
SEND I to motor driver
```

For real deployment, the trajectory (`theta_traj`) computed in Part 11
is sent to this loop. Each row is one timestep's reference.


In [ ]:
# ============================================================
# SIMULATED MOTOR CONTROL LOOP
# ============================================================

class PIDController:
    """
    PD controller per joint (I term omitted for simulation clarity).
    Real UR5 uses more sophisticated control, but this illustrates the concept.
    """
    def __init__(self, Kp, Kd, dt=0.008):
        """
        Kp : (6,) proportional gains [N.m/rad]
        Kd : (6,) derivative gains   [N.m.s/rad]
        dt : float  control timestep (8ms for UR5)
        """
        self.Kp = Kp
        self.Kd = Kd
        self.dt = dt
        self.prev_error = np.zeros(6)
    
    def compute(self, theta_ref, theta_actual, theta_dot_ref=None):
        """
        Compute torque command.
        Returns (6,) torque vector [N.m]
        """
        error     = theta_ref - theta_actual
        error_dot = (error - self.prev_error) / self.dt
        self.prev_error = error.copy()
        
        tau = self.Kp * error + self.Kd * error_dot
        return tau


def simulate_trajectory_execution(theta_traj, dt=0.008,
                                   noise_std=0.0005, friction_coef=0.1):
    """
    Simulate robot following the planned theta trajectory.
    Includes encoder noise and simplified joint friction.
    
    Parameters
    ----------
    theta_traj  : (N,6)  reference joint angle trajectory
    dt          : float  control timestep [s]
    noise_std   : float  encoder noise standard deviation [rad]
    friction_coef: float simplified friction coefficient
    
    Returns
    -------
    theta_actual : (N,6)  actual joint angles (with disturbances)
    torques      : (N,6)  applied joint torques
    track_errors : (N,6)  tracking errors per joint
    """
    n_steps = theta_traj.shape[0]
    
    # UR5 approximate PD gains [N.m/rad] and [N.m.s/rad]
    Kp = np.array([1500., 1500., 1000., 400., 400., 400.])
    Kd = np.array([150.,  150.,  100.,   40.,  40.,  40.])
    
    # UR5 approximate torque limits [N.m]
    TAU_MAX = np.array([150., 150., 150., 28., 28., 28.])
    
    pid = PIDController(Kp, Kd, dt)
    
    # Initial state: start at first reference
    theta_actual = theta_traj[0].copy()
    theta_vel    = np.zeros(6)
    
    results_actual = np.zeros((n_steps, 6))
    results_torque = np.zeros((n_steps, 6))
    results_error  = np.zeros((n_steps, 6))
    
    for k in range(n_steps):
        # Read encoder (with noise)
        theta_meas = theta_actual + np.random.randn(6) * noise_std
        
        # PID control
        tau = pid.compute(theta_traj[k], theta_meas)
        
        # Clamp to hardware torque limits
        tau = np.clip(tau, -TAU_MAX, TAU_MAX)
        
        # Simple dynamics: tau = I*theta_ddot + friction*theta_dot
        # Using approximate link inertia [kg.m^2]
        inertia  = np.array([3.0, 3.0, 1.5, 0.5, 0.5, 0.2])
        friction = friction_coef * np.sign(theta_vel)
        
        theta_acc = (tau - friction) / inertia
        theta_vel = theta_vel + theta_acc * dt
        theta_actual = theta_actual + theta_vel * dt
        
        # Clamp to joint limits
        theta_actual = np.clip(theta_actual, UR5_Q_MIN_SAFE, UR5_Q_MAX_SAFE)
        
        results_actual[k] = theta_actual
        results_torque[k] = tau
        results_error[k]  = theta_traj[k] - theta_actual
    
    return results_actual, results_torque, results_error


# Run simulation
print('Simulating trajectory execution at 125 Hz...')
theta_actual_sim, torques_sim, errors_sim = simulate_trajectory_execution(
    theta_traj, dt=0.008, noise_std=0.0002, friction_coef=0.05
)

# Compute end-effector error during simulation
ee_pos_errors = []
for k in range(len(theta_traj)):
    Q_ref = fk_dq(S, theta_traj[k], M_dq)
    Q_act = fk_dq(S, theta_actual_sim[k], M_dq)
    p_err, _ = pose_error_metrics(Q_act, Q_ref)
    ee_pos_errors.append(p_err * 1000)

print(f'Simulation complete.')
print(f'  Mean joint tracking error: {np.mean(np.abs(errors_sim))*1000:.4f} mrad')
print(f'  Max joint tracking error:  {np.max(np.abs(errors_sim))*1000:.4f} mrad')
print(f'  Mean EE position error:    {np.mean(ee_pos_errors):.4f} mm')
print(f'  Max EE position error:     {np.max(ee_pos_errors):.4f} mm')

In [ ]:
# ============================================================
# PLOT SIMULATION RESULTS
# ============================================================

time_sim = np.arange(len(theta_traj)) * 0.008   # 8ms timesteps

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Joint angle tracking
colors_j = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628']
for j in range(6):
    axes[0,0].plot(time_sim, np.degrees(theta_traj[:,j]),
                    '--', color=colors_j[j], alpha=0.7, linewidth=1.5, label=f'Ref J{j+1}')
    axes[0,0].plot(time_sim, np.degrees(theta_actual_sim[:,j]),
                    '-', color=colors_j[j], alpha=1.0, linewidth=1, label=f'Act J{j+1}')
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_ylabel('Joint Angle (deg)')
axes[0,0].set_title('Joint Tracking: Reference (--) vs Actual (-)')
axes[0,0].legend(loc='upper right', fontsize=7, ncol=2)
axes[0,0].grid(True, alpha=0.3)

# Tracking errors per joint
for j in range(6):
    axes[0,1].plot(time_sim, np.degrees(errors_sim[:,j]),
                    color=colors_j[j], linewidth=1.5, label=f'J{j+1}')
axes[0,1].set_xlabel('Time (s)'); axes[0,1].set_ylabel('Error (deg)')
axes[0,1].set_title('Joint Tracking Errors')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)
axes[0,1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# Applied torques
for j in range(6):
    axes[1,0].plot(time_sim, torques_sim[:,j],
                    color=colors_j[j], linewidth=1.5, label=f'J{j+1}')
axes[1,0].set_xlabel('Time (s)'); axes[1,0].set_ylabel('Torque (N.m)')
axes[1,0].set_title('Applied Joint Torques')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# End-effector position error
axes[1,1].plot(time_sim, ee_pos_errors, 'r-', linewidth=2)
axes[1,1].set_xlabel('Time (s)'); axes[1,1].set_ylabel('EE Position Error (mm)')
axes[1,1].set_title('End-Effector Position Error During Execution')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()